# 13. Supervised Learning: Linear Regression

## Algorithm Category
**Type**: Supervised Learning - Regression  
**Complexity**: Low  
**Use Case**: Predicting continuous numerical values

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand the mathematical foundation of linear regression
- Implement linear regression using scikit-learn
- Validate model assumptions and check residuals
- Evaluate regression performance using multiple metrics
- Perform hyperparameter tuning and cross-validation
- Interpret model coefficients and feature importance
- Apply linear regression to real-world datasets

## Historical Context

Linear regression is one of the oldest and most fundamental algorithms in statistics and machine learning. It was first developed by Francis Galton in the 1880s and later formalized by Karl Pearson and others. The method of least squares, which linear regression uses, was independently developed by Carl Friedrich Gauss and Adrien-Marie Legendre in the early 19th century.

**Key Papers/References:**
- Gauss, C.F. (1809). "Theoria motus corporum coelestium"
- Legendre, A.M. (1805). "Nouvelles méthodes pour la détermination des orbites des comètes"

## When to Use Linear Regression

Linear regression is appropriate when:
- The target variable is continuous (not categorical)
- There is a linear relationship between features and target
- Features are independent (or multicollinearity is addressed)
- The dataset is not too large (computational complexity is O(n²))
- Interpretability is important (coefficients are easy to understand)


## Theory & Mechanics

### Mathematical Foundation

Linear regression models the relationship between a dependent variable (target) y and one or more independent variables (features) X using a linear function:

**Simple Linear Regression (one feature):**
$$y = \beta_0 + \beta_1 x + \epsilon$$

**Multiple Linear Regression (multiple features):**
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_n x_n + \epsilon$$

Where:
- $y$ is the target variable
- $\beta_0$ is the intercept (bias term)
- $\beta_1, \beta_2, ..., \beta_n$ are the coefficients (weights) for each feature
- $x_1, x_2, ..., x_n$ are the feature values
- $\epsilon$ is the error term (residuals)

### How It Works

1. **Objective**: Find the coefficients that minimize the sum of squared residuals (Ordinary Least Squares - OLS)
2. **Cost Function**: Mean Squared Error (MSE)
   $$MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$
3. **Solution**: Closed-form solution using matrix algebra:
   $$\beta = (X^T X)^{-1} X^T y$$
4. **Prediction**: Once coefficients are learned, predictions are made by:
   $$\hat{y} = X\beta$$

### Key Assumptions

Linear regression assumes:
1. **Linearity**: Relationship between features and target is linear
2. **Independence**: Observations are independent of each other
3. **Homoscedasticity**: Constant variance of residuals across all feature values
4. **Normality**: Residuals are normally distributed
5. **No multicollinearity**: Features are not highly correlated with each other

### Limitations

- Cannot model non-linear relationships
- Sensitive to outliers
- Assumes linearity (may not capture complex patterns)
- Requires feature scaling for meaningful coefficient interpretation


## Implementation

Let's implement linear regression step by step using scikit-learn and our helper functions.


In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes, make_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Import our helper functions
from src.models.supervised import split_data, evaluate_regressor, cross_validate_model
from src.models.regression import (
    calculate_residuals, plot_residuals, check_residual_normality,
    calculate_goodness_of_fit, check_linearity_assumptions
)
from src.processing.preprocessing import scale_features
from src.utils.benchmarking import benchmark_model_training
from src.utils.traceability import extract_feature_importance_trace, save_traceability_data
from src.utils.validation import validate_model_output, check_cross_validation_stability

print("Libraries imported successfully!")


In [ ]:
# Load dataset - using Diabetes dataset (no download required)
diabetes = load_diabetes()
X = pd.DataFrame(diabetes.data, columns=diabetes.feature_names)
y = pd.Series(diabetes.target, name='Disease Progression')

print(f"Dataset Shape: {X.shape}")
print(f"Target Name: {diabetes.target_names[0] if hasattr(diabetes, 'target_names') else 'Disease Progression'}")
print(f"\nFirst few rows:")
print(X.head())
print(f"\nTarget statistics:")
print(y.describe())


In [ ]:
# Scale features for better coefficient interpretation
X_scaled, scaler = scale_features(X, fit=True)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = split_data(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")


In [ ]:
# Create and train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Intercept: {model.intercept_:.3f}")
print(f"\nCoefficients:")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)
print(coef_df)


In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Evaluate model
train_results = evaluate_regressor(model, X_train, y_train)
test_results = evaluate_regressor(model, X_test, y_test)

print("Training Performance:")
print(f"  RMSE: {train_results['rmse']:.3f}")
print(f"  MSE: {train_results['mse']:.3f}")

print("\nTest Performance:")
print(f"  RMSE: {test_results['rmse']:.3f}")
print(f"  MSE: {test_results['mse']:.3f}")

# Calculate R² score
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)
print(f"\nR² Score - Train: {r2_train:.3f}, Test: {r2_test:.3f}")


## Validation & Testing

Let's validate our model using multiple approaches: in-notebook assertions, cross-validation, and assumption checking.


In [ ]:
# Validation 1: Check model output validity
validation_result = validate_model_output(y_pred_test, y_test.values, task_type='regression')
print("Model Output Validation:")
print(f"  Valid: {validation_result['valid']}")
if 'mse' in validation_result:
    print(f"  MSE: {validation_result['mse']:.3f}")
    print(f"  RMSE: {validation_result['rmse']:.3f}")

# Assertions
assert validation_result['valid'], "Model predictions are invalid!"
assert test_results['rmse'] > 0, "RMSE should be positive"
assert r2_test >= 0, "R² score should be non-negative"
print("\n✓ Basic validation checks passed")


In [ ]:
# Validation 2: Cross-validation
cv_scores = cross_val_score(model, X_scaled, y, cv=5, scoring='neg_mean_squared_error')
cv_rmse = np.sqrt(-cv_scores)
cv_mean = cv_rmse.mean()
cv_std = cv_rmse.std()

print("Cross-Validation Results (5-fold):")
print(f"  Mean RMSE: {cv_mean:.3f} (+/- {cv_std:.3f})")
print(f"  Individual fold RMSEs: {cv_rmse}")

# Check cross-validation stability
stability = check_cross_validation_stability(cv_rmse, threshold=0.2)
print(f"\nCV Stability Check:")
print(f"  Coefficient of Variation: {stability['cv_coefficient']:.3f}")
print(f"  Is Stable: {stability['is_stable']}")

# Assertions
assert cv_mean > 0, "CV RMSE should be positive"
assert stability['is_stable'], "Cross-validation results are unstable!"
print("\n✓ Cross-validation checks passed")


In [ ]:
# Validation 3: Check regression assumptions
print("Checking Regression Assumptions:")
assumptions = check_linearity_assumptions(X_test.values, y_test.values, y_pred_test)
print(f"\n1. Residual-Prediction Correlation: {assumptions['residual_prediction_correlation']:.3f}")
print(f"   Homoscedasticity OK: {assumptions['homoscedasticity_ok']}")

print(f"\n2. Durbin-Watson Statistic: {assumptions['durbin_watson']:.3f}")
print(f"   Independence OK: {assumptions['independence_ok']}")

print(f"\n3. Normality Tests:")
normality = assumptions['normality']
print(f"   Shapiro-Wilk p-value: {normality['shapiro_wilk']['p_value']:.3f}")
print(f"   Is Normal (Shapiro): {normality['shapiro_wilk']['is_normal']}")
print(f"   D'Agostino p-value: {normality['dagostino']['p_value']:.3f}")
print(f"   Is Normal (D'Agostino): {normality['dagostino']['is_normal']}")

# Note: Assumptions may not always be perfectly met in practice
print("\nNote: Some assumptions may not be perfectly met, but model can still be useful.")


In [ ]:
# Validation 4: Goodness of fit metrics
goodness_of_fit = calculate_goodness_of_fit(y_test.values, y_pred_test)
print("Goodness of Fit Metrics:")
for metric, value in goodness_of_fit.items():
    print(f"  {metric}: {value:.3f}")

# Assertions
assert goodness_of_fit['r2_score'] >= 0, "R² should be non-negative"
assert goodness_of_fit['rmse'] > 0, "RMSE should be positive"
print("\n✓ Goodness of fit checks passed")


## Performance Benchmarking

Let's benchmark the model's performance in terms of training time, prediction speed, and compare with baseline.


In [ ]:
# Benchmark model training and prediction
benchmark_results = benchmark_model_training(
    LinearRegression(), X_train.values, y_train.values, 
    X_test.values, y_test.values
)

print("Performance Benchmark:")
print(f"  Training Time: {benchmark_results['training_time']:.4f} seconds")
print(f"  Prediction Time: {benchmark_results['prediction_time']:.4f} seconds")
print(f"  Predictions per Second: {benchmark_results['predictions_per_second']:.0f}")
print(f"  Test RMSE: {benchmark_results['test_rmse']:.3f}")
print(f"  Dataset Size: {benchmark_results['n_samples']} samples, {benchmark_results['n_features']} features")


In [ ]:
# Compare with baseline (mean predictor)
baseline_pred = np.full_like(y_test.values, y_train.mean())
baseline_rmse = np.sqrt(mean_squared_error(y_test.values, baseline_pred))

print("Baseline Comparison:")
print(f"  Baseline (Mean) RMSE: {baseline_rmse:.3f}")
print(f"  Linear Regression RMSE: {test_results['rmse']:.3f}")
print(f"  Improvement: {((baseline_rmse - test_results['rmse']) / baseline_rmse * 100):.1f}%")

assert test_results['rmse'] < baseline_rmse, "Model should outperform baseline!"
print("\n✓ Model outperforms baseline")


## Traceability

Let's extract and visualize feature importance, model coefficients, and create traceability records.


In [ ]:
# Extract feature importance (using coefficients)
feature_importance = extract_feature_importance_trace(model, feature_names=X.columns.tolist())
print("Feature Importance (by absolute coefficient):")
print(feature_importance)

# Visualize feature importance
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
plt.barh(range(len(feature_importance)), feature_importance['importance'], align='center')
plt.yticks(range(len(feature_importance)), feature_importance['feature'])
plt.xlabel('Absolute Coefficient Value')
plt.title('Feature Importance (Linear Regression Coefficients)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# Create traceability record
trace_data = {
    "model_type": "LinearRegression",
    "dataset": "Diabetes",
    "n_samples": len(X),
    "n_features": X.shape[1],
    "coefficients": dict(zip(X.columns, model.coef_)),
    "intercept": float(model.intercept_),
    "feature_importance": feature_importance.to_dict('records'),
    "performance_metrics": {
        "train_rmse": float(train_results['rmse']),
        "test_rmse": float(test_results['rmse']),
        "r2_train": float(r2_train),
        "r2_test": float(r2_test)
    },
    "cross_validation": {
        "mean_rmse": float(cv_mean),
        "std_rmse": float(cv_std)
    }
}

# Save traceability data
trace_path = save_traceability_data(trace_data, "linear_regression_trace")
print(f"Traceability data saved to: {trace_path}")


## Visualization & Diagnostics

Let's visualize the model's performance and check residuals.


In [ ]:
# Plot 1: Predicted vs Actual
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(y_test, y_pred_test, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Predicted vs Actual Values')
plt.grid(True, alpha=0.3)

# Plot 2: Residuals
residuals = calculate_residuals(y_test.values, y_pred_test)
plt.subplot(1, 2, 2)
plt.scatter(y_pred_test, residuals, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residual Plot')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Plot residuals with Q-Q plot for normality check
plot_residuals(y_test.values, y_pred_test, title="Linear Regression Residual Analysis")


## Real-World Application

Let's apply linear regression to a more complex scenario with hyperparameter tuning.


In [ ]:
# Note: LinearRegression doesn't have many hyperparameters, but we can use Ridge/Lasso
# For demonstration, let's create a synthetic dataset with known relationship
X_synthetic, y_synthetic = make_regression(
    n_samples=200, n_features=5, noise=20, random_state=42
)

X_syn_train, X_syn_test, y_syn_train, y_syn_test = train_test_split(
    X_synthetic, y_synthetic, test_size=0.2, random_state=42
)

# Train on synthetic data
model_syn = LinearRegression()
model_syn.fit(X_syn_train, y_syn_train)

y_syn_pred = model_syn.predict(X_syn_test)
syn_rmse = np.sqrt(mean_squared_error(y_syn_test, y_syn_pred))
syn_r2 = r2_score(y_syn_test, y_syn_pred)

print("Synthetic Dataset Results:")
print(f"  RMSE: {syn_rmse:.3f}")
print(f"  R²: {syn_r2:.3f}")

# Visualize
plt.figure(figsize=(8, 6))
plt.scatter(y_syn_test, y_syn_pred, alpha=0.6)
plt.plot([y_syn_test.min(), y_syn_test.max()], 
         [y_syn_test.min(), y_syn_test.max()], 'r--', lw=2)
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.title('Linear Regression on Synthetic Data')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Summary & Key Takeaways

### Key Concepts Learned

1. **Linear Regression Basics**
   - Models linear relationships between features and continuous target
   - Uses Ordinary Least Squares (OLS) to minimize sum of squared residuals
   - Provides interpretable coefficients

2. **Model Evaluation**
   - RMSE and MSE for regression error
   - R² score for goodness of fit
   - Cross-validation for robust performance estimation

3. **Assumptions & Diagnostics**
   - Check linearity, homoscedasticity, normality of residuals
   - Residual plots reveal model issues
   - Statistical tests validate assumptions

4. **Best Practices**
   - Scale features for meaningful coefficients
   - Use cross-validation to avoid overfitting
   - Check assumptions before trusting results
   - Compare with baseline models

### When to Use Linear Regression

✅ **Good for:**
- Continuous target variables
- Linear relationships
- Interpretability is important
- Small to medium datasets
- Baseline model for comparison

❌ **Not ideal for:**
- Non-linear relationships
- Very large datasets (use SGD variants)
- Highly correlated features (use regularization)
- Outliers (use robust regression)

### Next Steps

- Try **Ridge Regression** (L2 regularization) for multicollinearity
- Try **Lasso Regression** (L1 regularization) for feature selection
- Explore **Polynomial Regression** for non-linear relationships
- Consider **Elastic Net** for combining L1 and L2 regularization
